In [8]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from turtlesim.action import RotateAbsolute

In [ ]:
class TurtleRotateClient(Node):
    
    def __init__(self):
        super().__init__('turtle_rotate_client')
        self.client = ActionClient(self, RotateAbsolute, '/turtle1/rotate_absolute')
        self.get_logger().info('Action client started')

    def send_goal(self, theta):
        goal = RotateAbsolute.Goal()
        goal.theta = theta

        if not self.client.wait_for_server(timeout_sec=10.0):
            self.get_logger().info('Action server not available.')
            return

        goal_future = self.client.send_goal_async(goal, feedback_callback=self.feedback_callback)
        goal_future.add_done_callback(self.goal_response_callback)

    def goal_response_callback(self, future):
        handle = future.result()

        if not handle.accepted:
            self.get_logger().info('Goal rejected.')
            return
        
        self.get_logger().info('Goal accepted.')
        result_future = handle.get_result_async()
        result_future.add_done_callback(self.result_callback)

    def feedback_callback(self, feedback_msg):
        feedback = feedback_msg.feedback
        self.get_logger().info(f'Received feedback: remaining = {feedback.remaining:.2f}')

    def result_callback(self, future):
        result = future.result().result
        self.get_logger().info(f'Received result: delta = {result.delta:.2f}')
        self.destroy_node()   
        rclpy.shutdown()

In [10]:
rclpy.init()

client = TurtleRotateClient()

[INFO] [1761630248.577989600] [turtle_rotate_client]: Action client started


In [ ]:
client.send_goal(3.14)

In [ ]:
rclpy.spin(client)